In [1]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

StatementMeta(, a0476d99-c74f-422b-8b40-d0e80e9a4826, 3, Finished, Available, Finished, False)

DataFrame[]

In [6]:
from pyspark.sql.functions import current_timestamp, input_file_name

# Create bronze_customer_updates table
customer_updates_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/bronze/customer_updates.csv")
    .withColumn("bronze_load_timestamp", current_timestamp())
    .withColumn("source_file_name", input_file_name()))

customer_updates_df.write.mode("overwrite").format("delta").saveAsTable("bronze_customer_updates")


# Create bronze_order_updates table
order_updates_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/bronze/order_updates.csv")
    .withColumn("bronze_load_timestamp", current_timestamp())
    .withColumn("source_file_name", input_file_name()))

order_updates_df.write.mode("overwrite").format("delta").saveAsTable("bronze_order_updates")

print("Bronze incremental update tables created.")

StatementMeta(, a0476d99-c74f-422b-8b40-d0e80e9a4826, 8, Finished, Available, Finished, False)

Bronze incremental update tables created.


In [7]:
bronze_customer_updates = spark.table("bronze_customer_updates")

bronze_order_updates = spark.table("bronze_order_updates")

display(bronze_customer_updates)
display(bronze_order_updates)

StatementMeta(, a0476d99-c74f-422b-8b40-d0e80e9a4826, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f70ec48e-be69-4233-a1e7-c6e45a5d6d9c)

SynapseWidget(Synapse.DataFrame, 8401c4fe-9a74-4145-b458-57dc552843c0)

In [9]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col,current_timestamp, lit

silver_customers = DeltaTable.forName(spark, "silver.customers")

customer_updates = (
    bronze_customer_updates
    .withColumnRenamed("customer_zip_code_prefix", "customer_zip_code")
    .withColumn("customer_zip_code", col("customer_zip_code").cast("int"))
    .withColumn("silver_load_timestamp", current_timestamp())
    .withColumn("is_active", lit(True))
)

silver_customers.alias("target").merge(
    customer_updates.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdate(set={
    "customer_zip_code": "source.customer_zip_code",
    "customer_city": "source.customer_city",
    "customer_state": "source.customer_state",
    "silver_load_timestamp": "source.silver_load_timestamp",
    "is_active": "source.is_active"
}).whenNotMatchedInsert(values={
    "customer_id": "source.customer_id",
    "customer_zip_code": "source.customer_zip_code",
    "customer_city": "source.customer_city",
    "customer_state": "source.customer_state",
    "silver_load_timestamp": "source.silver_load_timestamp",
    "is_active": "source.is_active"
}).execute()

print("Customer MERGE completed.")

StatementMeta(, a0476d99-c74f-422b-8b40-d0e80e9a4826, 11, Finished, Available, Finished, False)

Customer MERGE completed.


In [10]:
spark.sql("""SELECT *FROM silver.customers
WHERE customer_id IN (
    '00012a2ce6f8dcda20d059ce98491703',
    'new_customer_001')""").show(truncate=False)

StatementMeta(, a0476d99-c74f-422b-8b40-d0e80e9a4826, 12, Finished, Available, Finished, False)

+--------------------------------+--------------------------------+-----------------+-----------------+--------------+--------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------+
|customer_id                     |customer_unique_id              |customer_zip_code|customer_city    |customer_state|bronze_load_timestamp     |source_file_name                                                                                                                                                                                 |silver_load_timestamp     |is_active|
+--------------------------------+--------------------------------+-----------------+-----------------+--------------+--------------------------+-----------------------------------------------------------------------------------------------------

**Delta History**

In [11]:
spark.sql("""DESCRIBE HISTORY silver.customers""").show(truncate=False)

StatementMeta(, a0476d99-c74f-422b-8b40-d0e80e9a4826, 13, Finished, Available, Finished, False)

+-------+-----------------------+------+--------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--